# Meme Organizer: 01. Лексический и Dense Zero-Shot поиск (Бейзлайны)

**Цель исследования:**
1. Построение воспроизводимого baseline ранжирования на чистом датасете `data/gold/gold_dataset.jsonl` (318 мемов, 1312 пар запросов).
2. Сравнение лексического поиска (**BM25 / SQLite FTS5**) и плотных нейросетевых энкодеров (**`rubert-tiny2`**, **`multilingual-e5-small`**).
3. Профилирование On-Device характеристик (задержка инференса на CPU в мс, размер ONNX INT8 в МБ).
4. Качественный анализ ошибок (**Top-15 Worst Queries**) и исследование эффекта гибридного слияния (Late Fusion).


In [ ]:
import json
from pathlib import Path

import mlflow
import numpy as np
import pandas as pd

from meme_organizer.data_splits.splitter import DatasetSplitter
from meme_organizer.evaluation.metrics import RetrievalEvaluator
from meme_organizer.retrieval.adapters.dense_text import DenseTextRetrieverAdapter
from meme_organizer.retrieval.baselines.bm25 import BM25RetrieverAdapter
from meme_organizer.retrieval.pipeline import RetrievalPipeline

# Фиксация путей
GOLD_DATASET_PATH = Path("../data/gold/gold_dataset.jsonl")
SPLIT_PATH = Path("../data/splits/splits_v1.json")

## 1. Проверка и загрузка данных без утечек (Leak-Free Split)

In [ ]:
with open(SPLIT_PATH) as f:
    split_info = json.load(f)

print(f"Корпус мемов: {len(split_info['corpus_hashes'])} документов")
print(f"Train пары:   {len(split_info['train_pairs'])} запросов")
print(f"Val пары:     {len(split_info['val_pairs'])} запросов")
print(f"Test пары:    {len(split_info['test_pairs'])} запросов")

## 2. Запуск Baseline 1: Лексический BM25 (аналог SQLite FTS5)

In [ ]:
pipeline = RetrievalPipeline()
bm25_adapter = BM25RetrieverAdapter(model_name="bm25-baseline")

bm25_metrics, bm25_run_id = pipeline.run(
    adapter=bm25_adapter,
    gold_dataset_path=GOLD_DATASET_PATH,
    split_path=SPLIT_PATH,
    split_name="test",
    run_name="bm25-baseline_test",
)

pd.DataFrame(
    [
        {
            "Model": "BM25 (Lexical)",
            "Recall@1": f"{bm25_metrics.recall_at_1:.4f}",
            "Recall@5": f"{bm25_metrics.recall_at_5:.4f}",
            "MRR@5": f"{bm25_metrics.mrr_at_5:.4f}",
            "NDCG@5": f"{bm25_metrics.ndcg_at_5:.4f}",
            "MLflow Run ID": bm25_run_id,
        }
    ]
)

## 3. Запуск Baseline 2: Dense Bi-Encoders (`rubert-tiny2` & `multilingual-e5-small`)

In [ ]:
rubert_adapter = DenseTextRetrieverAdapter(
    model_name="rubert-tiny2",
    hf_model_id="cointegrated/rubert-tiny2",
    max_length=64,
    onnx_dir=Path("../models/onnx"),
)

rubert_metrics, rubert_run_id = pipeline.run(
    adapter=rubert_adapter,
    gold_dataset_path=GOLD_DATASET_PATH,
    split_path=SPLIT_PATH,
    split_name="test",
    run_name="rubert-tiny2_test",
)

e5_adapter = DenseTextRetrieverAdapter(
    model_name="multilingual-e5-small",
    hf_model_id="intfloat/multilingual-e5-small",
    max_length=64,
    query_prefix="query: ",
    passage_prefix="passage: ",
    onnx_dir=Path("../models/onnx"),
)

e5_metrics, e5_run_id = pipeline.run(
    adapter=e5_adapter,
    gold_dataset_path=GOLD_DATASET_PATH,
    split_path=SPLIT_PATH,
    split_name="test",
    run_name="multilingual-e5-small_test",
)

## 4. Сравнительная сводная таблица из MLflow

In [ ]:
client = mlflow.tracking.MlflowClient("http://localhost:5000")
exp = client.get_experiment_by_name("meme-organizer-retrieval")
runs = client.search_runs(exp.experiment_id, order_by=["attribute.start_time DESC"], max_results=10)

leaderboard_rows = []
for r in runs:
    m = r.data.metrics
    leaderboard_rows.append(
        {
            "Run Name": r.data.tags.get("mlflow.runName", "unnamed"),
            "Recall@1": round(m.get("recall_at_1", 0.0), 4),
            "Recall@5": round(m.get("recall_at_5", 0.0), 4),
            "MRR@5": round(m.get("mrr_at_5", 0.0), 4),
            "Latency (ms)": round(m.get("cpu_latency_ms", 0.0), 2),
            "Size (MB)": round(m.get("model_size_mb", 0.0), 2),
            "Eval (s)": round(m.get("eval_inference_time_sec", 0.0), 3),
            "Run ID": r.info.run_id[:8],
        }
    )

pd.DataFrame(leaderboard_rows)

## 5. Анализ Top-15 Worst Queries (Качественный анализ ошибок)

In [ ]:
bm25_csv = client.download_artifacts(bm25_run_id, "error_analysis/worst_queries.csv")
df_worst_bm25 = pd.read_csv(bm25_csv)
print("=== Топ худших запросов для BM25 ===")
display(df_worst_bm25[["query", "target_rank"]].head(8))

e5_csv = client.download_artifacts(e5_run_id, "error_analysis/worst_queries.csv")
df_worst_e5 = pd.read_csv(e5_csv)
print("=== Топ худших запросов для E5-small ===")
display(df_worst_e5[["query", "target_rank"]].head(8))